<a href="https://colab.research.google.com/github/Miranics/TradesREvamp/blob/main/trade_review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Trade Review
Checks each gold and BTC trade against real 1-minute prices. Run the cells top to bottom.

**Step 1.** Make a data folder and check that Node.js is available.

In [ ]:
!mkdir -p data
!node -v

v20.19.0


**Step 2.** Upload your Exness CSV export. A file picker will appear.

In [ ]:
from google.colab import files
import shutil

uploaded = files.upload()
name = [n for n in uploaded if n.endswith('.csv')][0]
shutil.move(name, 'data/trades.csv')
print('Saved as data/trades.csv')

Saving 01_01_2007-05_10_2026.csv to 01_01_2007-05_10_2026.csv
Saved as data/trades.csv


**Step 3.** Download free 1-minute gold candles from Dukascopy (a few minutes).

In [ ]:
!npx --yes dukascopy-node -i xauusd -from 2026-01-27 -to 2026-10-07 -t m1 -f csv -dir data -fn xauusd_m1 -r 3 -s

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴----------------------------------------------------
7|░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░| 0%|█░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░| 1%|██░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░| 3%|██░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░| 4%|████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░| 7%|████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░| 8%|█████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░| 11%|██████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░| 12%|███████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░| 15%|███████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░| 16%|█████████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░| 19%|█████████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░| 20%|███████████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░| 23%|███████████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░| 24%|█████████████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░| 27%|█████████████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░| 28%|██████████████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░| 31%|██████████████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░| 32%|███████

**Step 4.** Download free 1-minute BTC candles from Dukascopy (a few minutes).

In [ ]:
import subprocess, os

!rm -f data/btcusd_part3.csv data/btcusd_part4.csv data/btcusd_part5.csv data/btcusd_part6.csv

ranges = [("2026-06-01", "2026-07-01"), ("2026-07-01", "2026-08-01"),
          ("2026-08-01", "2026-09-01"), ("2026-09-01", "2026-10-07")]

for i, (start, end) in enumerate(ranges, 3):
    name = f"btcusd_part{i}"
    print(f"Part {i}: {start} to {end} ... ", end="")
    cmd = (f"npx --yes dukascopy-node -i btcusd -from {start} -to {end} -t m1 -f csv "
           f"-dir data -fn {name} -bs 1 -bp 5000 -r 10 -rp 15000 -s")
    out = subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout
    if "File saved" in out:
        print("saved")
    else:
        print("FAILED, removed")
        if os.path.exists(f"data/{name}.csv"):
            os.remove(f"data/{name}.csv")

!ls -lh data

Part 3: 2026-06-01 to 2026-07-01 ... saved
Part 4: 2026-07-01 to 2026-08-01 ... FAILED, removed
Part 5: 2026-08-01 to 2026-09-01 ... FAILED, removed
Part 6: 2026-09-01 to 2026-10-07 ... FAILED, removed
total 25M
-rw-r--r-- 1 root root 4.0M Oct  5 20:58 btcusd_part1.csv
-rw-r--r-- 1 root root 3.8M Oct  5 20:59 btcusd_part2.csv
-rw-r--r-- 1 root root 1.9M Oct  5 21:23 btcusd_part3.csv
-rw-r--r-- 1 root root  37K Oct  5 20:52 trades.csv
-rw-r--r-- 1 root root  15M Oct  5 20:53 xauusd_m1.csv


**Step 5.** Load the analysis code.

In [ ]:
import glob

import pandas as pd

SYMBOLS = {"XAUUSDm": "data/xauusd*.csv", "BTCUSDm": "data/btcusd*.csv"}


def load_candles(pattern):
    files = glob.glob(pattern)
    if not files:
        return None
    df = pd.concat(pd.read_csv(f) for f in files)
    df["time"] = pd.to_datetime(df["timestamp"], unit="ms", utc=True)
    return df.drop_duplicates("time").set_index("time").sort_index()[["high", "low"]].astype(float)


def review(t, candles, horizon_h, rr):
    d = 1 if t.type == "buy" else -1
    entry, exit_ = t.opening_price, t.closing_price
    loss_dist = abs(entry - exit_)
    if pd.notna(t.take_profit) and t.take_profit > 0:
        tp, tp_src = t.take_profit, "your TP"
    else:
        tp, tp_src = entry + d * rr * loss_dist, f"{rr}R proxy"

    w = candles.loc[t.open_utc: t.open_utc + pd.Timedelta(hours=horizon_h)]
    if w.empty:
        return None

    adverse = (entry - w["low"]) if d == 1 else (w["high"] - entry)
    hit = (w["high"] >= tp) if d == 1 else (w["low"] <= tp)
    hit_idx = hit.idxmax() if hit.any() else None

    out = {
        "ticket": t.ticket, "symbol": t.symbol, "type": t.type, "open_utc": t.open_utc,
        "profit": t.profit, "close_reason": t.close_reason, "entry": entry, "tp_used": tp,
        "tp_source": tp_src, "loss_distance": round(loss_dist, 3),
        "tp_reached_within_horizon": hit_idx is not None,
        "hours_to_tp": round((hit_idx - t.open_utc).total_seconds() / 3600, 2) if hit_idx is not None else None,
        "stop_needed_to_survive": round(max(adverse.loc[:hit_idx].max(), 0), 3) if hit_idx is not None else None,
        "feed_gap": round(max(w["low"].iloc[0] - entry, entry - w["high"].iloc[0], 0), 3),
        "max_favorable_move": round((w["high"].max() - entry) if d == 1 else (entry - w["low"].min()), 3),
    }
    return out


def main(horizon=48, rr=3, trades_path="data/trades.csv"):

    trades = pd.read_csv(trades_path)
    trades = trades[trades["type"].isin(["buy", "sell"])].copy()
    trades["open_utc"] = pd.to_datetime(trades["opening_time_utc"], utc=True)

    results = []
    for sym, pattern in SYMBOLS.items():
        candles = load_candles(pattern)
        if candles is None:
            print(f"No candle file for {sym}, skipping")
            continue
        for t in trades[trades["symbol"] == sym].itertuples():
            r = review(t, candles, horizon, rr)
            if r:
                results.append(r)

    res = pd.DataFrame(results)
    res.to_csv("trade_review.csv", index=False)

    print("\n## Feed check: median gap between your entry and the data's first candle\n")
    print(res.groupby("symbol")["feed_gap"].median().to_string())

    losers = res[res["profit"] < 0]
    print(f"\n## Losing trades: did price later reach the target? (horizon {horizon:.0f}h)\n")
    summary = losers.groupby("symbol").agg(
        losing_trades=("ticket", "count"),
        target_reached=("tp_reached_within_horizon", "sum"),
        median_loss_distance=("loss_distance", "median"),
        median_stop_needed=("stop_needed_to_survive", "median"),
    )
    summary["target_reached_%"] = (100 * summary["target_reached"] / summary["losing_trades"]).round(1)
    print(summary.to_string())

    print("\n## Direction accuracy (all trades, target reached at any point, ignoring the stop)\n")
    acc = res.groupby("symbol")["tp_reached_within_horizon"].mean().mul(100).round(1)
    print(acc.to_string())

    saved = losers[losers["tp_reached_within_horizon"]]
    if not saved.empty:
        print("\n## Stop distance that would have survived to target (losing trades that later hit it)\n")
        print(saved.groupby("symbol")["stop_needed_to_survive"]
              .quantile([0.5, 0.75, 0.9]).unstack().round(2).to_string())

    print("\nFull per-trade detail saved to trade_review.csv")


**Step 6.** Run the review. Change `horizon` (hours to look ahead) or `rr` (target for trades without a TP) if you want.

In [ ]:
main(horizon=48, rr=3)


## Feed check: median gap between your entry and the data's first candle

symbol
BTCUSDm    0.015
XAUUSDm    0.000

## Losing trades: did price later reach the target? (horizon 48h)

         losing_trades  target_reached  median_loss_distance  median_stop_needed  target_reached_%
symbol                                                                                            
BTCUSDm              7               3               219.690            525.6000              42.9
XAUUSDm            191              76                 6.203             28.7355              39.8

## Direction accuracy (all trades, target reached at any point, ignoring the stop)

symbol
BTCUSDm    66.7
XAUUSDm    44.9

## Stop distance that would have survived to target (losing trades that later hit it)

           0.50    0.75     0.90
symbol                          
BTCUSDm  525.60  896.22  1118.60
XAUUSDm   28.74   54.23    79.23

Full per-trade detail saved to trade_review.csv


In [ ]:
import numpy as np

GRIDS = {"XAUUSDm": [5, 8, 10, 15, 20, 30, 40], "BTCUSDm": [100, 200, 300, 500, 800]}
RR = [1, 1.5, 2, 3, 4]
HORIZON_H = 48

trades = pd.read_csv("data/trades.csv")
trades = trades[trades["type"].isin(["buy", "sell"])].copy()
trades["open_utc"] = pd.to_datetime(trades["opening_time_utc"], utc=True)

rows = []
for sym, pattern in SYMBOLS.items():
    candles = load_candles(pattern)
    if candles is None or sym not in GRIDS:
        continue
    for t in trades[trades["symbol"] == sym].itertuples():
        w = candles.loc[t.open_utc: t.open_utc + pd.Timedelta(hours=HORIZON_H)]
        if len(w) < 60:
            continue
        d = 1 if t.type == "buy" else -1
        e = t.opening_price
        fav = ((w["high"] - e) if d == 1 else (e - w["low"])).cummax().to_numpy()
        adv = ((e - w["low"]) if d == 1 else (w["high"] - e)).cummax().to_numpy()
        last = ((w["high"].iloc[-1] + w["low"].iloc[-1]) / 2 - e) * d
        for stop in GRIDS[sym]:
            i_sl = np.searchsorted(adv, stop)
            for rr in RR:
                i_tp = np.searchsorted(fav, rr * stop)
                if i_sl == len(w) and i_tp == len(w):
                    r = max(min(last / stop, rr), -1)
                elif i_tp < i_sl:
                    r = rr
                else:
                    r = -1
                rows.append((sym, stop, rr, r))

g = pd.DataFrame(rows, columns=["symbol", "stop", "rr", "R"])
summary = g.groupby(["symbol", "stop", "rr"]).agg(
    trades=("R", "size"), win_rate=("R", lambda x: round((x > 0).mean() * 100, 1)),
    avg_R=("R", lambda x: round(x.mean(), 3)), total_R=("R", lambda x: round(x.sum(), 1))).reset_index()
summary.to_csv("grid_results.csv", index=False)

for sym in summary["symbol"].unique():
    s = summary[summary["symbol"] == sym]
    print(f"\n## {sym}: average R per trade (rows = stop in $, columns = target in R)\n")
    print(s.pivot(index="stop", columns="rr", values="avg_R").to_string())
    print(f"\n## {sym}: win rate %\n")
    print(s.pivot(index="stop", columns="rr", values="win_rate").to_string())
    print(f"\n## {sym}: top 5 combinations\n")
    print(s.sort_values("avg_R", ascending=False).head(5).to_string(index=False))


## BTCUSDm: average R per trade (rows = stop in $, columns = target in R)

rr      1.0    1.5   2.0    3.0    4.0
stop                                  
100  -0.167  0.042  0.00 -0.333 -0.167
200   0.000  0.042  0.00  0.333  0.667
300   0.000  0.250  0.50  1.000  0.667
500   0.333  0.458  0.75  0.333  0.200
800   0.167  0.042  0.00 -0.009 -0.080

## BTCUSDm: win rate %

rr     1.0   1.5   2.0   3.0   4.0
stop                              
100   41.7  41.7  33.3  16.7  16.7
200   50.0  41.7  33.3  33.3  33.3
300   50.0  50.0  50.0  50.0  33.3
500   66.7  58.3  58.3  33.3  25.0
800   58.3  41.7  33.3  33.3  33.3

## BTCUSDm: top 5 combinations

 symbol  stop  rr  trades  win_rate  avg_R  total_R
BTCUSDm   300 3.0      12      50.0  1.000     12.0
BTCUSDm   500 2.0      12      58.3  0.750      9.0
BTCUSDm   300 4.0      12      33.3  0.667      8.0
BTCUSDm   200 4.0      12      33.3  0.667      8.0
BTCUSDm   300 2.0      12      50.0  0.500      6.0

## XAUUSDm: average R per trade (ro

In [ ]:
c = load_candles("data/xauusd*.csv")
c = c[c.index.dayofweek < 5]
days = []
for day, d in c.groupby(c.index.date):
    def win(h1, h2):
        return d[(d.index.hour >= h1) & (d.index.hour < h2)]
    asia, london, ny = win(0, 7), win(7, 12), win(13, 21)
    if len(london) < 200 or len(ny) < 300:
        continue
    open_ny, close_ny = (ny["high"].iloc[0] + ny["low"].iloc[0]) / 2, (ny["high"].iloc[-1] + ny["low"].iloc[-1]) / 2
    up = close_ny > open_ny
    lh, ll = london["high"].max(), london["low"].min()
    took_low, took_high = ny["low"].min() < ll, ny["high"].max() > lh
    if up:
        t_ext = ny["high"].idxmax()
        retrace = open_ny - ny.loc[:t_ext, "low"].min()
        swept_opp = ny.loc[:t_ext, "low"].min() < ll
    else:
        t_ext = ny["low"].idxmin()
        retrace = ny.loc[:t_ext, "high"].max() - open_ny
        swept_opp = ny.loc[:t_ext, "high"].max() > lh
    days.append(dict(day=day, up=up, move=abs(close_ny - open_ny), retrace=max(retrace, 0),
                     swept_opposite_first=swept_opp, took_london_low=took_low, took_london_high=took_high,
                     close_vs_london_low=close_ny > ll, close_vs_london_high=close_ny < lh))
s = pd.DataFrame(days)
s.to_csv("sweep_stats.csv", index=False)

print(f"Days analysed: {len(s)}  (NY = 13:00-21:00 UTC, London = 07:00-12:00 UTC)\n")
print("## If your bias for the NY session was right\n")
print(f"NY swept the opposite London extreme before the real move: {s.swept_opposite_first.mean()*100:.1f}% of days")
print(f"It just continued without sweeping: {(1-s.swept_opposite_first.mean())*100:.1f}% of days\n")
print("Retracement against you from the NY open before the move ($):")
print(s.retrace.quantile([0.5, 0.75, 0.9]).round(2).to_string(), "\n")
print("## After NY sweeps a London extreme, does price reverse back inside it by the close?\n")
lo, hi = s[s.took_london_low], s[s.took_london_high]
print(f"NY took London low on {len(lo)} days -> closed back above it: {lo.close_vs_london_low.mean()*100:.1f}%")
print(f"NY took London high on {len(hi)} days -> closed back below it: {hi.close_vs_london_high.mean()*100:.1f}%")
print(f"\nMedian NY session move ($): {s.move.median():.2f}")

Days analysed: 177  (NY = 13:00-21:00 UTC, London = 07:00-12:00 UTC)

## If your bias for the NY session was right

NY swept the opposite London extreme before the real move: 44.6% of days
It just continued without sweeping: 55.4% of days

Retracement against you from the NY open before the move ($):
0.50    13.10
0.75    23.09
0.90    35.16 

## After NY sweeps a London extreme, does price reverse back inside it by the close?

NY took London low on 111 days -> closed back above it: 45.0%
NY took London high on 107 days -> closed back below it: 51.4%

Median NY session move ($): 27.38


In [ ]:
import glob
import numpy as np
import pandas as pd

TIMEFRAMES = [1, 5, 15]
RR_LIST = [1.5, 2, 3]
SPREAD = 0.30
BUFFER = 0.50
DISPLACEMENT = 1.5

raw = pd.concat(pd.read_csv(f) for f in glob.glob("data/xauusd*.csv"))
raw["time"] = pd.to_datetime(raw["timestamp"], unit="ms", utc=True)
m1 = raw.drop_duplicates("time").set_index("time").sort_index()[["open", "high", "low", "close"]].astype(float)
m1 = m1[m1.index.dayofweek < 5]


def swing_points(b, side):
    s = b["low"] if side == "low" else b["high"]
    if side == "low":
        mask = (s < s.shift(1)) & (s < s.shift(2)) & (s < s.shift(-1)) & (s < s.shift(-2))
    else:
        mask = (s > s.shift(1)) & (s > s.shift(2)) & (s > s.shift(-1)) & (s > s.shift(-2))
    return mask


def find_setup(b, lh, ll):
    body = (b["close"] - b["open"]).abs()
    avg_body = body.rolling(20, min_periods=5).median().shift(1)
    ny = b[(b.index.hour >= 13) & (b.index.hour < 17)]
    for t in ny.index:
        i = b.index.get_loc(t)
        bar = b.iloc[i]
        for d, swept in ((-1, bar["high"] > lh), (1, bar["low"] < ll)):
            if not swept:
                continue
            prior = b.iloc[max(0, i - 40):i]
            sw = swing_points(b.iloc[max(0, i - 42):i + 1], "low" if d == -1 else "high").reindex(prior.index).fillna(False)
            if not sw.any():
                continue
            level = prior.loc[sw[sw].index[-1], "low" if d == -1 else "high"]
            extreme = bar["high"] if d == -1 else bar["low"]
            for j in range(i + 1, min(i + 30, len(b))):
                x = b.iloc[j]
                extreme = max(extreme, x["high"]) if d == -1 else min(extreme, x["low"])
                broke = x["close"] < level if d == -1 else x["close"] > level
                strong = body.iloc[j] >= DISPLACEMENT * (avg_body.iloc[j] or np.inf)
                if broke and strong:
                    for k in range(j + 1, i, -1):
                        if k >= len(b) or k - 2 < 0:
                            continue
                        a, c = b.iloc[k - 2], b.iloc[k]
                        if d == -1 and a["low"] > c["high"]:
                            return dict(dir=d, entry=c["high"], stop=extreme + BUFFER, ready=b.index[min(k + 1, len(b) - 1)])
                        if d == 1 and a["high"] < c["low"]:
                            return dict(dir=d, entry=c["low"], stop=extreme - BUFFER, ready=b.index[min(k + 1, len(b) - 1)])
                    return None
            return None
    return None


def simulate(s, day_m1, rr):
    d, entry, stop = s["dir"], s["entry"], s["stop"]
    risk = abs(entry - stop) + SPREAD
    after = day_m1[(day_m1.index >= s["ready"]) & (day_m1.index.hour < 21)]
    filled = False
    for t, x in after.iterrows():
        if not filled:
            if (d == -1 and x["high"] >= stop) or (d == 1 and x["low"] <= stop):
                return None
            if (d == -1 and x["high"] >= entry) or (d == 1 and x["low"] <= entry):
                filled = True
                if t.hour >= 20:
                    return None
            continue
        target = entry + d * rr * risk
        hit_sl = x["high"] >= stop if d == -1 else x["low"] <= stop
        hit_tp = x["low"] <= target if d == -1 else x["high"] >= target
        if hit_sl:
            return -1.0
        if hit_tp:
            return rr
    if not filled:
        return None
    last = after["close"].iloc[-1]
    return max(min(d * (last - entry) / risk - SPREAD / risk, rr), -1.0)


rows = []
for day, d1 in m1.groupby(m1.index.date):
    london = d1[(d1.index.hour >= 7) & (d1.index.hour < 12)]
    if len(london) < 200:
        continue
    lh, ll = london["high"].max(), london["low"].min()
    for tf in TIMEFRAMES:
        b = d1 if tf == 1 else d1.resample(f"{tf}min").agg(
            {"open": "first", "high": "max", "low": "min", "close": "last"}).dropna()
        s = find_setup(b, lh, ll)
        if not s:
            continue
        for rr in RR_LIST:
            r = simulate(s, d1, rr)
            if r is not None:
                rows.append(dict(day=day, tf=tf, rr=rr, dir=s["dir"], stop_size=abs(s["entry"] - s["stop"]), R=r))

res = pd.DataFrame(rows)
res.to_csv("mss_backtest.csv", index=False)
out = res.groupby(["tf", "rr"]).agg(
    trades=("R", "size"), win_rate=("R", lambda x: round((x > 0).mean() * 100, 1)),
    avg_R=("R", lambda x: round(x.mean(), 3)), total_R=("R", lambda x: round(x.sum(), 1)),
    median_stop=("stop_size", lambda x: round(x.median(), 2)))
print("## Sweep of London extreme -> displacement MSS -> FVG entry (gold, NY 13:00-17:00 UTC)\n")
print(out.to_string())

## Sweep of London extreme -> displacement MSS -> FVG entry (gold, NY 13:00-17:00 UTC)

        trades  win_rate  avg_R  total_R  median_stop
tf rr                                                
1  1.5      49      38.8 -0.049     -2.4        10.86
   2.0      49      30.6 -0.137     -6.7        10.86
   3.0      49      26.5 -0.113     -5.6        10.86
5  1.5      47      53.2  0.220     10.3        24.97
   2.0      47      53.2  0.245     11.5        24.97
   3.0      47      53.2  0.357     16.8        24.97
15 1.5      35      48.6  0.162      5.7        32.43
   2.0      35      48.6  0.155      5.4        32.43
   3.0      35      48.6  0.158      5.5        32.43


In [ ]:
r5 = res[(res.tf == 5) & (res.rr == 3)].copy()
r5["month"] = pd.to_datetime(r5["day"]).dt.to_period("M")
print("## 5-min, 3R: by month\n")
print(r5.groupby("month").agg(trades=("R", "size"), win_rate=("R", lambda x: round((x > 0).mean() * 100, 1)), total_R=("R", "sum")).round(2).to_string())
print("\n## 5-min, 3R: buys vs sells\n")
print(r5.groupby("dir").agg(trades=("R", "size"), win_rate=("R", lambda x: round((x > 0).mean() * 100, 1)), total_R=("R", "sum")).round(2).rename(index={1: "buy", -1: "sell"}).to_string())
print(f"\nFull wins (+3R): {(r5.R == 3).sum()}   Full losses (-1R): {(r5.R == -1).sum()}   Closed at 21:00: {((r5.R > -1) & (r5.R < 3)).sum()}")

## 5-min, 3R: by month

         trades  win_rate  total_R
month                             
2026-02       4      50.0     1.26
2026-03       7      71.4     5.76
2026-04       3      66.7     2.71
2026-05       4      25.0    -0.80
2026-06       6      16.7    -3.53
2026-07       5      60.0     2.46
2026-08      10      70.0     6.81
2026-09       7      42.9     0.07
2026-10       1     100.0     2.01

## 5-min, 3R: buys vs sells

      trades  win_rate  total_R
dir                            
sell      22      63.6    18.24
buy       25      44.0    -1.48

Full wins (+3R): 4   Full losses (-1R): 14   Closed at 21:00: 29


In [ ]:
daily_close = m1["close"].resample("1D").last().dropna()
trend = np.sign(daily_close - daily_close.shift(5)).shift(1)

rows = []
for disp in [1.2, 1.5, 2.0]:
    DISPLACEMENT = disp
    for day, d1 in m1.groupby(m1.index.date):
        london = d1[(d1.index.hour >= 7) & (d1.index.hour < 12)]
        if len(london) < 200:
            continue
        b = d1.resample("5min").agg({"open": "first", "high": "max", "low": "min", "close": "last"}).dropna()
        s = find_setup(b, london["high"].max(), london["low"].min())
        if not s:
            continue
        tr = trend.get(pd.Timestamp(day, tz="UTC"), np.nan)
        for rr, label in ((3, "3R target"), (10, "no target, exit 21:00")):
            r = simulate(s, d1, rr)
            if r is not None:
                rows.append(dict(displacement=disp, exit=label, dir=s["dir"],
                                 with_trend=(s["dir"] == tr), R=r))
DISPLACEMENT = 1.5

rb = pd.DataFrame(rows)
agg = dict(trades=("R", "size"), win_rate=("R", lambda x: round((x > 0).mean() * 100, 1)),
           avg_R=("R", lambda x: round(x.mean(), 3)), total_R=("R", lambda x: round(x.sum(), 1)))
print("## 5-min model: sensitivity to displacement strength and exit rule\n")
print(rb.groupby(["displacement", "exit"]).agg(**agg).to_string())
print("\n## 5-min model, displacement 1.5: with vs against the 5-day trend\n")
print(rb[rb.displacement == 1.5].groupby(["exit", "with_trend"]).agg(**agg).to_string())

## 5-min model: sensitivity to displacement strength and exit rule

                                    trades  win_rate  avg_R  total_R
displacement exit                                                   
1.2          3R target                  50      54.0  0.415     20.7
             no target, exit 21:00      50      54.0  0.531     26.5
1.5          3R target                  47      53.2  0.357     16.8
             no target, exit 21:00      47      53.2  0.440     20.7
2.0          3R target                  41      51.2  0.315     12.9
             no target, exit 21:00      41      51.2  0.410     16.8

## 5-min model, displacement 1.5: with vs against the 5-day trend

                                  trades  win_rate  avg_R  total_R
exit                  with_trend                                  
3R target             False           24      58.3  0.567     13.6
                      True            23      47.8  0.136      3.1
no target, exit 21:00 False           24    

In [ ]:
def simulate_managed(s, day_m1, be_at=None, adds=()):
    d, entry, stop = s["dir"], s["entry"], s["stop"]
    risk = abs(entry - stop)
    after = day_m1[(day_m1.index >= s["ready"]) & (day_m1.index.hour < 21)]
    units, filled, todo = [], False, list(adds)
    for t, x in after.iterrows():
        if not filled:
            if (d == -1 and x["high"] >= stop) or (d == 1 and x["low"] <= stop):
                return None
            if (d == -1 and x["high"] >= entry) or (d == 1 and x["low"] <= entry):
                if t.hour >= 20:
                    return None
                filled, units = True, [entry]
            continue
        hit_sl = x["high"] >= stop if d == -1 else x["low"] <= stop
        if hit_sl:
            return sum(d * (stop - e) / risk for e in units) - len(units) * SPREAD / risk
        best = (entry - x["low"]) if d == -1 else (x["high"] - entry)
        if be_at and best >= be_at * risk:
            stop = entry
        while todo and best >= todo[0] * risk:
            units.append(entry + d * todo.pop(0) * risk)
            stop = entry
    if not filled:
        return None
    last = after["close"].iloc[-1]
    return sum(d * (last - e) / risk for e in units) - len(units) * SPREAD / risk


variants = {"1. Baseline (hold to 21:00)": dict(),
            "2. Breakeven at +1R": dict(be_at=1),
            "3. Add 1 at +1R": dict(adds=(1,)),
            "4. Add at +1R and +2R": dict(adds=(1, 2))}

DISPLACEMENT = 1.5
rows = []
for day, d1 in m1.groupby(m1.index.date):
    london = d1[(d1.index.hour >= 7) & (d1.index.hour < 12)]
    if len(london) < 200:
        continue
    b = d1.resample("5min").agg({"open": "first", "high": "max", "low": "min", "close": "last"}).dropna()
    s = find_setup(b, london["high"].max(), london["low"].min())
    if not s:
        continue
    for name, kw in variants.items():
        r = simulate_managed(s, d1, **kw)
        if r is not None:
            rows.append(dict(variant=name, R=r))

mv = pd.DataFrame(rows)
print("## 5-min model: trade management comparison (R = multiples of original risk)\n")
print(mv.groupby("variant").agg(trades=("R", "size"),
      win_rate=("R", lambda x: round((x > 0).mean() * 100, 1)),
      avg_R=("R", lambda x: round(x.mean(), 3)), total_R=("R", lambda x: round(x.sum(), 1)),
      best_trade=("R", lambda x: round(x.max(), 1)), worst_trade=("R", lambda x: round(x.min(), 1))).to_string())

## 5-min model: trade management comparison (R = multiples of original risk)

                             trades  win_rate  avg_R  total_R  best_trade  worst_trade
variant                                                                               
1. Baseline (hold to 21:00)      47      53.2  0.448     21.1         7.4         -1.0
2. Breakeven at +1R              47      42.6  0.278     13.1         7.4         -1.0
3. Add 1 at +1R                  47      40.4  0.474     22.3        13.7         -1.0
4. Add at +1R and +2R            47      40.4  0.648     30.4        19.1         -1.0


**Step 7.** Download the per-trade detail file.

In [ ]:
files.download('trade_review.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>